# GOLD LAYER: Valuation Analysis

Identify overvalued ("bubble") stocks by comparing:
- **Valuation:** P/E ratio, Price-to-Sales
- **Growth:** Revenue growth rate
- **Profitability:** Net profit margin
- **Risk:** Bubble risk score (0-100)

In [ ]:
import pandas as pd
import numpy as np

# Load silver layer outputs
fundamentals = pd.read_csv('/workspaces/bubbleindex/silver/combined_fundamentals.csv')
prices = pd.read_csv('/workspaces/bubbleindex/silver/combined_stockprices.csv')

# Convert dates to datetime
fundamentals['Date'] = pd.to_datetime(fundamentals['Date'])
prices['Date'] = pd.to_datetime(prices['Date'])

print(f"✓ Loaded: {fundamentals['ticker'].nunique()} companies")
print(f"✓ Fundamentals: {len(fundamentals)} rows")
print(f"✓ Prices: {len(prices)} rows")

In [ ]:
# Export snapshot results
export_data = snapshot[['ticker', 'close_price', 'PE_Ratio', 'Price_to_Sales', 'Revenue growth', 'Net Margin', 'Bubble_Risk_Score']].copy()
export_data.columns = ['Ticker', 'Stock_Price', 'PE_Ratio', 'Price_to_Sales', 'Growth_Pct', 'Margin_Pct', 'Risk_Score']
export_data = export_data.sort_values('Risk_Score', ascending=False)
export_data.to_csv('/workspaces/bubbleindex/gold/valuation_results.csv', index=False)

print("\n" + "="*80)
print("GOLD LAYER COMPLETE ✓")
print("="*80)
print("✓ Part 1: Analyzed fundamentals trends independently (years of data)")
print("✓ Part 2: Assessed valuation snapshot (latest data alignment)")
print("✓ Merged both perspectives for complete investment view")
print("✓ Exported: /workspaces/bubbleindex/gold/valuation_results.csv")
print("="*80)
print("\nNote: Fundamentals data is fiscal-year (annual), prices are daily (current).")
print("This 3-month gap is OK: fundamentals show business health, prices show market sentiment.")
print("="*80)

## Part 4: Export Results

In [ ]:
print("\n" + "="*80)
print("KEY INSIGHTS & INVESTMENT PERSPECTIVE")
print("="*80)

# Insights from snapshot (valuation metrics)
top_growth = snapshot.nlargest(1, 'Revenue growth')
top_margin = snapshot.nlargest(1, 'Net Margin')
lowest_pe = snapshot.nsmallest(1, 'PE_Ratio')
highest_risk = snapshot.nlargest(1, 'Bubble_Risk_Score')
lowest_risk = snapshot.nsmallest(1, 'Bubble_Risk_Score')

print(f"\n🚀 GROWTH LEADER: {top_growth['ticker'].values[0]}")
print(f"   → Revenue Growth: {top_growth['Revenue growth'].values[0]*100:+.1f}%")
print(f"   → Valuation: P/E {top_growth['PE_Ratio'].values[0]:.1f}x")

print(f"\n💰 PROFITABILITY LEADER: {top_margin['ticker'].values[0]}")
print(f"   → Net Margin: {top_margin['Net Margin'].values[0]*100:.1f}%")
print(f"   → Valuation: P/E {top_margin['PE_Ratio'].values[0]:.1f}x")

print(f"\n💎 BEST VALUE (Cheapest): {lowest_pe['ticker'].values[0]}")
print(f"   → P/E Ratio: {lowest_pe['PE_Ratio'].values[0]:.1f}x")
print(f"   → Growth: {lowest_pe['Revenue growth'].values[0]*100:+.1f}%")

print(f"\n⚠️  HIGHEST BUBBLE RISK: {highest_risk['ticker'].values[0]}")
print(f"   → Risk Score: {highest_risk['Bubble_Risk_Score'].values[0]:.0f}/100")
print(f"   → P/E: {highest_risk['PE_Ratio'].values[0]:.1f}x | Growth: {highest_risk['Revenue growth'].values[0]*100:+.1f}%")

print(f"\n✅ SAFEST INVESTMENT: {lowest_risk['ticker'].values[0]}")
print(f"   → Risk Score: {lowest_risk['Bubble_Risk_Score'].values[0]:.0f}/100")
print(f"   → Balanced: Moderate P/E with stable growth")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print("• FUNDAMENTALS TRENDS (Part 1): Show business quality over time")
print("• VALUATION SNAPSHOT (Part 2): Show current price relative to latest fundamentals")
print("• Both perspectives needed: Historical health + current market pricing")
print("="*80)

## Part 3: Integrated Insights

In [ ]:
# Get latest fundamentals and prices for each company
latest_fundamentals = fundamentals.sort_values('Date').groupby('ticker').tail(1).reset_index(drop=True)
latest_prices = prices.sort_values('Date').groupby('ticker').tail(1).reset_index(drop=True)

# Merge latest metrics
snapshot = latest_fundamentals.merge(latest_prices, on='ticker', how='inner')

# Calculate valuation multiples
snapshot['PE_Ratio'] = snapshot['close_price'] / snapshot['Diluted EPS']
snapshot['Price_to_Sales'] = (snapshot['close_price'] * 1000000) / snapshot['Total Revenue']

# Bubble Risk Score (0-100): High P/E + Slowing Growth
snapshot['Bubble_Risk_Score'] = (
    (snapshot['PE_Ratio'] / snapshot['PE_Ratio'].max()) * 50 +      # 50 pts for high P/E
    (np.maximum(0, -snapshot['Revenue growth']) / 0.3) * 50          # 50 pts for negative/slowing growth
).clip(0, 100)

print("\n" + "="*80)
print("VALUATION SNAPSHOT - As of March 2026")
print("="*80)

# Print detailed valuation table
val_table = snapshot[['ticker', 'close_price', 'PE_Ratio', 'Price_to_Sales', 'Revenue growth', 'Net Margin', 'Bubble_Risk_Score']].copy()
val_table.columns = ['Ticker', 'Stock Price', 'P/E', 'P/Sales', 'Growth %', 'Margin %', 'Risk Score']
val_table = val_table.sort_values('Risk Score', ascending=False)

for col in ['P/E', 'P/Sales', 'Growth %', 'Margin %', 'Risk Score']:
    if col == 'Growth %':
        val_table[col] = val_table[col].apply(lambda x: f'{x*100:+.1f}%')
    elif col == 'Margin %':
        val_table[col] = val_table[col].apply(lambda x: f'{x*100:.1f}%')
    elif col == 'Risk Score':
        val_table[col] = val_table[col].apply(lambda x: f'{x:.0f}/100')
    elif col in ['P/E', 'P/Sales']:
        val_table[col] = val_table[col].apply(lambda x: f'{x:.1f}x')
    else:
        val_table[col] = val_table[col].apply(lambda x: f'${x:.2f}')

print(val_table.to_string(index=False))

print("\n" + "="*80)
print("BUBBLE RISK RANKING")
print("="*80)

for idx, row in snapshot.sort_values('Bubble_Risk_Score', ascending=False).iterrows():
    ticker = row['ticker']
    risk = row['Bubble_Risk_Score']
    pe = row['PE_Ratio']
    growth = row['Revenue growth'] * 100
    status = "🔴 HIGH RISK" if risk > 60 else "🟡 MODERATE" if risk > 40 else "🟢 LOW RISK"
    print(f"{ticker:8} | P/E: {pe:5.1f}x | Growth: {growth:+6.1f}% | Score: {risk:5.0f}/100 | {status}")

print("\nSCORING: 50 points for high P/E multiple + 50 points for slowing/negative growth")
print("="*80)

## PART 2: Valuation Snapshot Analysis
**Current Point-in-Time Assessment** (Latest Fundamentals + Current Prices)

Compare valuation multiples and identify bubble risk using most recent data alignment.

In [ ]:
print("\n" + "="*80)
print("PART 1: FUNDAMENTALS TREND ANALYSIS")
print("="*80)

# Sort by ticker and date to see trends over time
fund_trends = fundamentals.sort_values(['ticker', 'Date']).copy()

print("\n📊 PROFITABILITY TRENDS (Net Margin %)")
print("-" * 80)
for ticker in sorted(fund_trends['ticker'].unique()):
    ticker_data = fund_trends[fund_trends['ticker'] == ticker][['Date', 'Net Margin']].copy()
    ticker_data['Net Margin'] = ticker_data['Net Margin'] * 100
    ticker_data = ticker_data.sort_values('Date')
    
    if len(ticker_data) > 1:
        margin_change = ticker_data['Net Margin'].iloc[-1] - ticker_data['Net Margin'].iloc[0]
        trend_dir = "↑" if margin_change > 0 else "↓" if margin_change < 0 else "→"
        print(f"{ticker}: {ticker_data['Net Margin'].iloc[-1]:.1f}% (latest) | {trend_dir} {margin_change:+.1f}pp change")
    else:
        print(f"{ticker}: {ticker_data['Net Margin'].values[0]:.1f}% (limited data)")

print("\n📈 REVENUE GROWTH TRENDS (YoY %)")
print("-" * 80)
for ticker in sorted(fund_trends['ticker'].unique()):
    ticker_data = fund_trends[fund_trends['ticker'] == ticker][['Date', 'Revenue growth']].sort_values('Date')
    growth_values = ticker_data['Revenue growth'].dropna() * 100
    
    if len(growth_values) > 0:
        avg_growth = growth_values.mean()
        latest_growth = growth_values.iloc[-1] if len(growth_values) > 0 else 0
        print(f"{ticker}: Latest {latest_growth:+.1f}% | Avg: {avg_growth:+.1f}%")

print("\n💪 R&D INTENSITY TRENDS (% of Revenue)")
print("-" * 80)
for ticker in sorted(fund_trends['ticker'].unique()):
    ticker_data = fund_trends[fund_trends['ticker'] == ticker][['Date', 'R&D Intensity']].sort_values('Date')
    rd_values = ticker_data['R&D Intensity'].dropna() * 100
    
    if len(rd_values) > 0:
        latest_rd = rd_values.iloc[-1]
        avg_rd = rd_values.mean()
        print(f"{ticker}: Latest {latest_rd:.1f}% | Avg: {avg_rd:.1f}% | {'Innovation-focused' if latest_rd > 15 else 'Balanced'}")

print("\n" + "="*80)

## PART 1: Fundamentals Trend Analysis
**Time Period Analysis** (Independent of Stock Prices)

Analyze profitability, growth, and leverage metrics over multiple years to understand business fundamentals trajectory.

## 1. Load & Prepare Data